# D&D Beyond Homebrew Sync

Sync homebrew monsters from Google Sheets to D&D Beyond.

## Setup Instructions

1. **Get your D&D Beyond cookies**:
   - Open D&D Beyond in your browser
   - Log in to your account
   - Open Developer Tools (F12)
   - Go to Network tab
   - Visit any D&D Beyond page
   - Find the request, right-click → Copy → Copy as cURL
   - Extract the Cookie header value

2. **Get security tokens**:
   - Go to https://www.dndbeyond.com/homebrew/creations/create-monster/create
   - Open Developer Tools → Network tab
   - Fill out the form (don't submit yet)
   - Right-click → Inspect on the form
   - Find the hidden `security-token` and `authenticity-token` fields

3. **Configure below** with your cookies and tokens

## Important Notes

- **Security tokens expire**: You'll need to refresh them periodically
- **Rate limiting**: D&D Beyond may throttle requests
- **Master subscription**: Some features require a Master-tier subscription


## 1. Setup and Imports


In [1]:
import pandas as pd
import requests
import json
import time
import re
import os
from typing import Dict, List, Optional
from dotenv import load_dotenv
from datetime import datetime
from urllib.parse import urlencode
from bs4 import BeautifulSoup
from pathlib import Path
import sys

# Ensure the repo root is on sys.path so FiveETools can be imported
repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from FiveETools.core.Helpers.gsheets_client import ContentSheetsClient
from FiveETools.core.fantasy import monster as monsters

# Use the service account credentials bundled with FiveETools
fantasy_sheets = ContentSheetsClient("fantasy", credentials_path=str(repo_root / "FiveETools" / "key.json"))

from DNDBeyond.core.Helpers.DnDBeyondAPI import DnDBeyondAPI
from DNDBeyond.core.Helpers.converter import _MONSTER_FIELD_IDS

def _title_case(name: str) -> str:
    return " ".join([part.capitalize() for part in (name or "").split()])

def _snake_title(name: str) -> str:
    return "_".join([part.capitalize() for part in (name or "").split()]).replace(',', '')

def _monster_avatar_paths(monster_name: str, repo_root: Path) -> dict:
    title = _title_case(monster_name)
    snake_title = _snake_title(monster_name)
    small = repo_root / "assets" / "tokens" / f"{snake_title}.png"
    large = repo_root / "assets" / "art" / "Monsters" / "fantasy" / f"{title}.alt.png"
    return {"small": small, "large": large}

def _open_avatar_files(avatar_paths: dict) -> dict:
    avatar_files = {}
    small = avatar_paths.get("small")
    large = avatar_paths.get("large")
    if small and small.exists():
        avatar_files[_MONSTER_FIELD_IDS["avatar_small"]] = open(small, "rb")
    else:
        print(f"⚠️  Missing small avatar: {small}")
    if large and large.exists():
        avatar_files[_MONSTER_FIELD_IDS["avatar_large"]] = open(large, "rb")
    else:
        print(f"⚠️  Missing large avatar: {large}")
    return avatar_files

def _close_avatar_files(avatar_files: dict) -> None:
    for f in avatar_files.values():
        try:
            f.close()
        except Exception:
            pass


MONSTER_CREATE_HTML = repo_root / "DNDBeyond" / "pages" / "Create - Create a Monster - Creations - Homebrew - D&D Beyond.html"

def _get_required_form_fields(html_path: Path) -> set:
    if not html_path.exists():
        return set()
    text = html_path.read_text(encoding="utf-8", errors="ignore")
    return set(re.findall(r'name="([^"]+)"[^>]*data-validation-required="true"', text))

def _get_required_file_fields(html_path: Path) -> set:
    if not html_path.exists():
        return set()
    text = html_path.read_text(encoding="utf-8", errors="ignore")
    required = set()
    patterns = [
        r'req[^>]*>Small Avatar</div>.*?<input[^>]+type="file"[^>]+name="([^"]+)"',
        r'req[^>]*>Large Avatar</div>.*?<input[^>]+type="file"[^>]+name="([^"]+)"',
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.S)
        if match:
            required.add(match.group(1))
    return required

REQUIRED_FORM_FIELDS = _get_required_form_fields(MONSTER_CREATE_HTML)
REQUIRED_FILE_FIELDS = _get_required_file_fields(MONSTER_CREATE_HTML)

def _validate_monster_form(form_data: dict, avatar_paths: dict) -> dict:
    missing_fields = [
        key for key in REQUIRED_FORM_FIELDS
        if key not in form_data or str(form_data.get(key, "")).strip() == ""
    ]
    missing_files = []
    if _MONSTER_FIELD_IDS["avatar_small"] in REQUIRED_FILE_FIELDS:
        if not avatar_paths.get("small") or not avatar_paths["small"].exists():
            missing_files.append(_MONSTER_FIELD_IDS["avatar_small"])
    if _MONSTER_FIELD_IDS["avatar_large"] in REQUIRED_FILE_FIELDS:
        if not avatar_paths.get("large") or not avatar_paths["large"].exists():
            missing_files.append(_MONSTER_FIELD_IDS["avatar_large"])
    return {"missing_fields": missing_fields, "missing_files": missing_files}
print("✓ Imports loaded")


✓ Imports loaded


## 2. D&D Beyond Authentication

Configure your D&D Beyond cookies and security tokens.

**Note**: The sync will automatically track DDB monster IDs in your Google Sheets by adding them to a "DDB" column.


In [2]:
# ============================================
# D&D Beyond Configuration
# ============================================

load_dotenv()

DDB_BASE_URL = os.getenv("DDB_BASE_URL")
DDB_COOKIES = os.getenv("DDB_COOKIES")
DDB_SECURITY_TOKEN = os.getenv("DDB_SECURITY_TOKEN")
DDB_AUTHENTICITY_TOKEN = os.getenv("DDB_AUTHENTICITY_TOKEN")
REQUEST_VERIFICATION_TOKEN = os.getenv("REQUEST_VERIFICATION_TOKEN")
DDB_USER_ID = os.getenv("DDB_USER_ID")
DDB_USERNAME = os.getenv("DDB_USERNAME")

# ============================================
# Setup Session
# ============================================

session = requests.Session()

if DDB_COOKIES:
    session.headers.update({
        "Cookie": DDB_COOKIES,
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br, zstd",
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:147.0) Gecko/20100101 Firefox/147.0",
        "Referer": f"{DDB_BASE_URL}/homebrew/creations/create-monster/create",
        "Origin": DDB_BASE_URL,
        "Sec-Fetch-Dest": "document",
        "Sec-Fetch-Mode": "navigate",
        "Sec-Fetch-Site": "same-origin",
        "Sec-Fetch-User": "?1",
        "Upgrade-Insecure-Requests": "1"
    })

    # Refresh monster field IDs from the live create page
    try:
        from DNDBeyond.core.Helpers.converter import refresh_monster_field_ids_from_html_text
        response = session.get(f"{DDB_BASE_URL}/homebrew/creations/create-monster/create", timeout=30)
        refresh_monster_field_ids_from_html_text(response.text)
        # Update required field lists from the same HTML
        global REQUIRED_FORM_FIELDS, REQUIRED_FILE_FIELDS
        REQUIRED_FORM_FIELDS = set(re.findall(r'name="([^"]+)"[^>]*data-validation-required="true"', response.text))
        REQUIRED_FILE_FIELDS = set()
        required_patterns = [
            r'req[^>]*>Small Avatar</div>.*?<input[^>]+type="file"[^>]+name="([^"]+)"',
            r'req[^>]*>Large Avatar</div>.*?<input[^>]+type="file"[^>]+name="([^"]+)"',
        ]
        for pattern in required_patterns:
            match = re.search(pattern, response.text, re.S)
            if match:
                REQUIRED_FILE_FIELDS.add(match.group(1))
        print("✓ Refreshed monster field IDs from live page")
    except Exception as exc:
        print(f"⚠️  Could not refresh monster field IDs: {exc}")
    print("✓ Using cookie authentication")
    print(f"✓ Base URL: {DDB_BASE_URL}")
    print(f"✓ User: {DDB_USERNAME} (ID: {DDB_USER_ID})")

    if DDB_SECURITY_TOKEN and DDB_AUTHENTICITY_TOKEN:
        print("✓ Security tokens configured")
    else:
        print("⚠️  Security tokens not set - you'll need these to create monsters!")
else:
    print("✗ ERROR: No cookies provided!")
    print("   Please set DDB_COOKIES with your browser session cookies")


✓ Refreshed monster field IDs from live page
✓ Using cookie authentication
✓ Base URL: https://www.dndbeyond.com
✓ User: Rockoteque (ID: 110746825)
✓ Security tokens configured


## 3. Load Monster Data from Google Sheets


In [3]:
# Load fantasy monsters
print("Loading monsters from Google Sheets...")

monsters = monsters.monster_list

print(f"✓ Loaded {len(monsters)} monsters")
print(f"\nSample monster:")
if monsters:
    sample = monsters[0]
    print(f"  Name: {sample.get('name', 'N/A')}")
    print(f"  Type: {sample.get('type', 'N/A')}")
    print(f"  CR: {sample.get('challenge_rating', 'N/A')}")

# Also load the raw DataFrame to check for existing DDB IDs
print("\nLoading spreadsheet data for DDB ID tracking...")
MONSTERS_GID = "736393386"  # Fantasy monsters sheet GID
MONSTER_NAME_COLUMN = "Name"

df_monsters = fantasy_sheets.get_sheet(MONSTERS_GID)
print(f"✓ Loaded spreadsheet with {len(df_monsters)} rows")

# Ensure DDB column exists
print("\nEnsuring 'DDB' column exists in spreadsheet...")
try:
    ddb_col_idx = fantasy_sheets.ensure_column_exists(MONSTERS_GID, "DDB")
    print(f"✓ 'DDB' column exists at index {ddb_col_idx}")

    # Check if any monsters already have DDB IDs
    df_monsters = fantasy_sheets.get_sheet(MONSTERS_GID)  # Reload after ensuring column
    if 'DDB' in df_monsters.columns:
        existing_ddb_ids = df_monsters[df_monsters['DDB'].notna()]['DDB'].tolist()
        print(f"✓ Found {len(existing_ddb_ids)} monsters already synced to D&D Beyond")
    else:
        print("  'DDB' column created")
except Exception as e:
    print(f"⚠️  Could not ensure DDB column: {e}")
    print("  You may need to check your Google Sheets credentials")
    print("  Continuing without spreadsheet tracking...")


Loading monsters from Google Sheets...
✓ Loaded 98 monsters

Sample monster:
  Name: Pavani
  Type: monstrosity
  CR: N/A

Loading spreadsheet data for DDB ID tracking...
✓ Loaded spreadsheet with 1034 rows

Ensuring 'DDB' column exists in spreadsheet...
✓ 'DDB' column exists at index 74
✓ Found 0 monsters already synced to D&D Beyond


## 4. D&D Beyond API Helper

Helper class for interacting with D&D Beyond's homebrew creation endpoints.


In [4]:
ddb_api = DnDBeyondAPI(session, DDB_SECURITY_TOKEN, DDB_AUTHENTICITY_TOKEN, REQUEST_VERIFICATION_TOKEN)
print("✓ D&D Beyond API helper initialized")


✓ D&D Beyond API helper initialized


In [5]:
from DNDBeyond.core.Helpers import convert_monster_to_ddb

print("✓ D&D Beyond converter loaded")


✓ D&D Beyond converter loaded


## 5.5 Duplicate Checking (HTML Parsing)

The `monsters.list()` method uses HTML parsing to fetch existing homebrew monsters.

**How it works:**

D&D Beyond uses Server-Side Rendering (SSR) and embeds monster data in the HTML response rather than providing a separate API endpoint. The method:

1. Fetches `/my-creations?filter-type=779871897` (monster type filter)
2. Parses the HTML to find all elements with `class="list-row-homebrew-creation-Monster"`
3. Extracts monster IDs and names from the `data-slug` attributes
   - Format: `data-slug="6057264-test-monster"`
   - Parsed to: `{"id": "6057264", "name": "test monster"}`

**Filter Status Options:**
- `filter-status=1` - Published monsters only
- `filter-status=2` - Draft monsters only
- Use both parameters to fetch all monsters

You can test the duplicate checking in the next cell.


In [6]:
# Test fetching existing monsters with HTML parsing
print("Testing monsters.list() method...\n")

try:
    existing_monsters = ddb_api.monsters.list()

    if existing_monsters:
        print(f"✓ Successfully parsed {len(existing_monsters)} existing homebrew monsters\n")
        print("Found monsters:")
        for i, monster in enumerate(existing_monsters, 1):
            monster_name = monster.get('name', 'Unknown')
            monster_id = monster.get('id', 'Unknown')
            print(f"{i:2d}. {monster_name} (ID: {monster_id})")
            print(f"    URL: {DDB_BASE_URL}/homebrew/creations/monsters/{monster_id}")

        print(f"\n✓ Duplicate checking is ready to use!")
        print(f"  When syncing, monsters with matching names will be skipped.")
    else:
        print("⚠️  No monsters found")
        print("\nPossible reasons:")
        print("1. You don't have any published homebrew monsters yet")
        print("2. Your session cookies may have expired")
        print("3. The HTML structure may have changed")
        print("\nTo include draft monsters, update the params in monsters.list():")
        print('  params = {"filter-type": "779871897", "filter-status": "2"}')

except Exception as e:
    print(f"✗ Error fetching monsters: {e}")
    print("\nThis could mean:")
    print("1. Session cookies have expired - update DDB_COOKIES")
    print("2. D&D Beyond's HTML structure changed")
    print("3. Network or authentication issue")
    print("\nYou can still sync without duplicate checking.")
    import traceback
    traceback.print_exc()


Testing monsters.list() method...

✓ Successfully parsed 5 existing homebrew monsters

Found monsters:
 1. locutus (ID: 3567503)
    URL: https://www.dndbeyond.com/homebrew/creations/monsters/3567503
 2. myconid swordsman (ID: 4783356)
    URL: https://www.dndbeyond.com/homebrew/creations/monsters/4783356
 3. quercus assassin (ID: 3035696)
    URL: https://www.dndbeyond.com/homebrew/creations/monsters/3035696
 4. test monster (ID: 6057264)
    URL: https://www.dndbeyond.com/homebrew/creations/monsters/6057264
 5. vandrexamon (ID: 3567518)
    URL: https://www.dndbeyond.com/homebrew/creations/monsters/3567518

✓ Duplicate checking is ready to use!
  When syncing, monsters with matching names will be skipped.


## 6. Sync Monsters to D&D Beyond

**Configuration Options:**
- `DRY_RUN`: Set to `False` to actually create/update monsters (default: `False`)
- `BATCH_SIZE`: Set to `None` to sync all monsters, or a number to limit (e.g., `10` for testing)
- `DELAY`: Seconds between requests to avoid rate limiting (default: `2`)
- `VERBOSE`: Detailed logging for debugging (default: `True`)
- `SKIP_EXISTING`: Set to `True` to skip monsters that already exist (default: `True`)
  - When `True`: Monsters with DDB IDs in spreadsheet or found on D&D Beyond will be skipped (if `UPDATE_EXISTING=False`)
  - When `False`: Will attempt to create monsters even if they already exist (may create duplicates or fail)
- `UPDATE_EXISTING`: Set to `True` to UPDATE monsters when DDB ID exists (default: `True`)
  - When `True` and `SKIP_EXISTING=False`: Monsters with DDB IDs will be UPDATED instead of skipped
  - This is useful when you rename monsters in Google Sheets - the sync will update the monster name on D&D Beyond


In [7]:
# ============================================
# SYNC MONSTERS (WITH UPDATE SUPPORT)
# ============================================

# Import normalize_ddb_id helper to handle pandas float conversion
from DNDBeyond.core.Helpers import normalize_ddb_id

DRY_RUN = False  # Set to False to actually create/update monsters
BATCH_SIZE = 1  # Set to None for all monsters, or a number to limit (e.g., 10)
DELAY = 1  # Seconds between requests
VERBOSE = True  # Set to True for detailed logging
SKIP_EXISTING = True  # Set to False to attempt creating monsters even if they already exist
UPDATE_EXISTING = True  # Set to True to UPDATE monsters when DDB ID exists

print("=" * 60)
if DRY_RUN:
    print("DRY RUN - No monsters will be created/updated")
else:
    print("⚠️  LIVE MODE - Monsters WILL be created/updated!")
print("=" * 60)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Batch size: {'ALL MONSTERS' if BATCH_SIZE is None else BATCH_SIZE}")
print(f"Delay between requests: {DELAY}s")
print(f"Verbose logging: {'ON' if VERBOSE else 'OFF'}")
print(f"Skip existing: {'YES' if SKIP_EXISTING else 'NO (will attempt to create anyway)'}")
print(f"Update existing: {'YES (will update when DDB ID exists)' if UPDATE_EXISTING else 'NO'}")
print()

# Select monsters to sync
monsters_to_sync = monsters if BATCH_SIZE is None else monsters[:BATCH_SIZE]
print(f"Total monsters to process: {len(monsters_to_sync)}")
print()

# Check spreadsheet for existing DDB IDs first
print("Checking spreadsheet for existing DDB IDs...")
df_monsters = fantasy_sheets.get_sheet(MONSTERS_GID)
monster_to_ddb_id = {}
if 'DDB' in df_monsters.columns and MONSTER_NAME_COLUMN in df_monsters.columns:
    for _, row in df_monsters.iterrows():
        monster_name = row.get(MONSTER_NAME_COLUMN)
        ddb_id = row.get('DDB')
        if monster_name and pd.notna(ddb_id):
            normalized_id = normalize_ddb_id(ddb_id)
            if normalized_id:
                monster_to_ddb_id[monster_name] = normalized_id
    print(f"✓ Found {len(monster_to_ddb_id)} monsters with DDB IDs in spreadsheet")
else:
    print("  No DDB IDs found in spreadsheet")

# Fetch existing monsters from D&D Beyond HTML (backup check)
print("\nFetching existing homebrew monsters from D&D Beyond...")
if not DRY_RUN:
    try:
        existing_monsters = ddb_api.monsters.list()
        print(f"✓ Found {len(existing_monsters)} existing homebrew monsters\n")
    except Exception as e:
        print(f"⚠️  Could not fetch existing monsters: {e}")
        print("Using spreadsheet DDB IDs only...\n")
        existing_monsters = []
else:
    existing_monsters = []

# Refresh required fields from live page before validation
if not DRY_RUN:
    try:
        from DNDBeyond.core.Helpers.converter import refresh_monster_field_ids_from_html_text
        response = session.get(f"{DDB_BASE_URL}/homebrew/creations/create-monster/create", timeout=30)
        refresh_monster_field_ids_from_html_text(response.text)
        REQUIRED_FORM_FIELDS = set(re.findall(r'name="([^"]+)"[^>]*data-validation-required="true"', response.text))
        REQUIRED_FILE_FIELDS = set()
        required_patterns = [
            r'req[^>]*>Small Avatar</div>.*?<input[^>]+type="file"[^>]+name="([^"]+)"',
            r'req[^>]*>Large Avatar</div>.*?<input[^>]+type="file"[^>]+name="([^"]+)"',
        ]
        for pattern in required_patterns:
            match = re.search(pattern, response.text, re.S)
            if match:
                REQUIRED_FILE_FIELDS.add(match.group(1))
        print("✓ Refreshed required monster fields from live page")
    except Exception as exc:
        print(f"⚠️  Could not refresh required monster fields: {exc}")
results = {
    'created': 0,
    'updated': 0,
    'skipped': 0,
    'errors': 0,
    'details': []
}

# Track updates to write back to spreadsheet
spreadsheet_updates = []

for i, monster in enumerate(monsters_to_sync, 1):
    monster_name = monster.get('name', 'Unnamed')
    print(f"[{i}/{len(monsters_to_sync)}] Processing: {monster_name}")

    try:
        # Convert to D&D Beyond format first (needed for both create and update)
        if VERBOSE:
            print(f"  → Converting monster to D&D Beyond format...")

        try:
            ddb_form = convert_monster_to_ddb(monster)
            if VERBOSE:
                print(f"    Type: {ddb_form.get('monster-type')}, Size: {ddb_form.get('size')}")
                print(f"    CR: {ddb_form.get('challenge-rating')}")
        except Exception as conv_error:
            print(f"  ✗ Conversion failed: {conv_error}")
            if VERBOSE:
                import traceback
                print(f"    Traceback:\n{traceback.format_exc()}")
            results['errors'] += 1
            results['details'].append({
                'title': monster_name,
                'action': 'error',
                'success': False,
                'error': f"Conversion error: {str(conv_error)}",
                'error_type': 'conversion',
                'monster_data': {k: str(v)[:100] for k, v in monster.items()}
            })
            continue

        # Check if monster already has DDB ID in spreadsheet
        existing_id = monster_to_ddb_id.get(monster_name)

        if existing_id:
            # Monster has a DDB ID - decide whether to UPDATE or SKIP
            if UPDATE_EXISTING and not SKIP_EXISTING:
                # UPDATE mode: Update the existing monster
                if DRY_RUN:
                    print(f"  → DRY RUN: Would update monster (ID: {existing_id})")
                    results['details'].append({
                        'title': monster_name,
                        'action': 'dry_run_update',
                        'success': True,
                        'id': existing_id
                    })
                    results['updated'] += 1
                else:
                    print(f"  → Updating existing monster (ID: {existing_id})")
                    if VERBOSE:
                        print(f"    URL: {DDB_BASE_URL}/homebrew/creations/monsters/{existing_id}")

                    slug = DnDBeyondAPI.create_slug(monster_name)
                    avatar_paths = _monster_avatar_paths(monster_name, repo_root)
                    avatar_files = _open_avatar_files(avatar_paths)
                    updated = ddb_api.monsters.update(existing_id, slug, {'form_data': ddb_form, 'files': avatar_files})
                    _close_avatar_files(avatar_files)

                    if updated:
                        print(f"  ✓ Updated monster successfully")
                        results['updated'] += 1
                        results['details'].append({
                            'title': monster_name,
                            'action': 'updated',
                            'success': True,
                            'id': existing_id
                        })
                    else:
                        print(f"  ✗ Failed to update monster")
                        error_info = {
                            'title': monster_name,
                            'action': 'error',
                            'success': False,
                            'error_type': 'update_failed',
                            'id': existing_id
                        }

                        if ddb_api.last_response is not None:
                            try:
                                error_info['last_response'] = {
                                    'status_code': ddb_api.last_response.status_code,
                                    'reason': ddb_api.last_response.reason,
                                    'url': getattr(ddb_api.last_response, 'url', None),
                                    'text_preview': ddb_api.last_response.text[:1000],
                                }
                            except Exception:
                                pass
                        if ddb_api.last_error:
                            error_info['error'] = ddb_api.last_error
                        results['errors'] += 1
                        results['details'].append(error_info)

                    # Rate limiting
                    if i < len(monsters_to_sync):
                        time.sleep(DELAY)
            else:
                # SKIP mode: Just skip the monster
                print(f"  ✓ Already synced (DDB ID: {existing_id} from spreadsheet)")
                if VERBOSE:
                    print(f"    URL: {DDB_BASE_URL}/homebrew/creations/monsters/{existing_id}")
                results['skipped'] += 1
                results['details'].append({
                    'title': monster_name,
                    'action': 'skipped',
                    'success': True,
                    'id': existing_id,
                    'reason': 'already_in_spreadsheet'
                })
            continue

        # No DDB ID in spreadsheet - check D&D Beyond directly (backup check)
        if SKIP_EXISTING and not DRY_RUN and existing_monsters:
            existing_id = ddb_api.monsters.find_by_name(monster_name, existing_monsters)
            if existing_id:
                print(f"  ⚠️  Monster exists on D&D Beyond (ID: {existing_id})")
                if VERBOSE:
                    print(f"    URL: {DDB_BASE_URL}/homebrew/creations/monsters/{existing_id}")
                print(f"    Updating spreadsheet with ID...")

                spreadsheet_updates.append({
                    'match_value': monster_name,
                    'update_column': 'DDB',
                    'update_value': existing_id
                })

                results['skipped'] += 1
                results['details'].append({
                    'title': monster_name,
                    'action': 'skipped',
                    'success': True,
                    'id': existing_id,
                    'reason': 'already_exists_ddb'
                })
                continue

        # No existing ID found - CREATE new monster
        if DRY_RUN:
            print(f"  → DRY RUN: Would create monster")
            results['details'].append({
                'title': monster_name,
                'action': 'dry_run_create',
                'success': True
            })
            results['created'] += 1
        else:
            # Create monster
            if VERBOSE:
                print(f"  → Creating monster on D&D Beyond...")

            avatar_paths = _monster_avatar_paths(monster_name, repo_root)
            validation = _validate_monster_form(ddb_form, avatar_paths)
            if validation["missing_fields"] or validation["missing_files"]:
                print("  ✗ Missing required fields/files for create")
                if validation["missing_fields"]:
                    print(f"    Missing fields: {validation['missing_fields']}")
                if validation["missing_files"]:
                    print(f"    Missing files: {validation['missing_files']}")
                results["errors"] += 1
                results["details"].append({
                    "title": monster_name,
                    "action": "error",
                    "success": False,
                    "error_type": "validation_failed",
                    "missing_fields": validation["missing_fields"],
                    "missing_files": validation["missing_files"],
                })
                continue
            avatar_files = _open_avatar_files(avatar_paths)
            monster_id = ddb_api.monsters.create({'form_data': ddb_form, 'files': avatar_files})
            _close_avatar_files(avatar_files)

            if monster_id:
                print(f"  ✓ Created: ID {monster_id}")
                print(f"    URL: {DDB_BASE_URL}/homebrew/creations/monsters/{monster_id}")

                if VERBOSE:
                    print(f"    Adding to spreadsheet update queue...")

                # Add to batch update
                spreadsheet_updates.append({
                    'match_value': monster_name,
                    'update_column': 'DDB',
                    'update_value': monster_id
                })

                results['created'] += 1
                results['details'].append({
                    'title': monster_name,
                    'action': 'created',
                    'success': True,
                    'id': monster_id
                })

                # Add to existing list for subsequent checks
                existing_monsters.append({'name': monster_name, 'id': monster_id})
            else:
                print(f"  ✗ Failed to create monster")

                # Log detailed error information
                error_info = {
                    'title': monster_name,
                    'action': 'error',
                    'success': False,
                    'error_type': 'creation_failed'
                }

                if ddb_api.last_response is not None:
                    try:
                        error_info['last_response'] = {
                            'status_code': ddb_api.last_response.status_code,
                            'reason': ddb_api.last_response.reason,
                            'url': getattr(ddb_api.last_response, 'url', None),
                            'text_preview': ddb_api.last_response.text[:1000],
                        }
                    except Exception:
                        pass

                # Include last error details
                if ddb_api.last_error:
                    if isinstance(ddb_api.last_error, dict):
                        error_info['error'] = ddb_api.last_error
                        if 'status_code' in ddb_api.last_error:
                            print(f"    HTTP {ddb_api.last_error['status_code']}: {ddb_api.last_error.get('reason', 'Unknown')}")
                        elif 'exception_type' in ddb_api.last_error:
                            print(f"    Exception: {ddb_api.last_error['exception_type']}")
                    else:
                        error_info['error'] = str(ddb_api.last_error)
                        print(f"    Error: {ddb_api.last_error}")
                else:
                    error_info['error'] = 'Unknown error - no details available'
                    print(f"    Error: Unknown (no details)")

                # Include converted data for debugging
                if VERBOSE:
                    error_info['converted_data'] = {
                        k: str(v)[:200] if isinstance(v, str) else v
                        for k, v in ddb_form.items()
                    }

                results['errors'] += 1
                results['details'].append(error_info)

            # Rate limiting
            if i < len(monsters_to_sync):
                time.sleep(DELAY)

    except Exception as e:
        print(f"  ✗ Unexpected error: {e}")
        if VERBOSE:
            import traceback
            print(f"    Traceback:\n{traceback.format_exc()}")

        results['errors'] += 1
        results['details'].append({
            'title': monster_name,
            'action': 'error',
            'success': False,
            'error': str(e),
            'error_type': 'unexpected_exception',
            'traceback': traceback.format_exc() if VERBOSE else None
        })

# Write DDB IDs back to spreadsheet
if spreadsheet_updates and not DRY_RUN:
    print('\n' + '=' * 60)
    print('Updating Google Sheets with DDB IDs...')
    print('=' * 60)
    try:
        update_results = fantasy_sheets.batch_update_cells_by_row_match(
            MONSTERS_GID,
            MONSTER_NAME_COLUMN,
            spreadsheet_updates
        )
        success_count = sum(1 for v in update_results.values() if v)
        print(f"✓ Updated {success_count}/{len(spreadsheet_updates)} monsters in spreadsheet")

        # Log any failures
        failed = [k for k, v in update_results.items() if not v]
        if failed:
            print(f"⚠️  Failed to update {len(failed)} monsters:")
            for monster_name in failed[:5]:
                print(f"  - {monster_name}")
            if len(failed) > 5:
                print(f"  ... and {len(failed) - 5} more")
    except Exception as e:
        print(f"✗ Error updating spreadsheet: {e}")
        print('  Monster IDs were created but not recorded in spreadsheet')
        if VERBOSE:
            import traceback
            print(f"  Traceback:\n{traceback.format_exc()}")

print()
print('=' * 60)
print('SYNC COMPLETE')
print('=' * 60)
if DRY_RUN:
    print(f"DRY RUN: Would create {results['created']} monsters")
    print(f"DRY RUN: Would update {results['updated']} monsters")
    print('\nSet DRY_RUN = False to actually create/update')
else:
    print(f"✓ Created: {results['created']}")
    print(f"✓ Updated: {results['updated']}")
    print(f"⚠️  Skipped (already exist): {results['skipped']}")
    print(f"✗ Errors: {results['errors']}")

    if results['errors'] > 0:
        print(f"\nℹ️  Check the log file for detailed error information")
print('=' * 60)
print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Save log with enhanced error details
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
log_file = f"out/monster_sync_log_{timestamp}.json"
log_dir = os.path.dirname(log_file)
os.makedirs(log_dir, exist_ok=True)
with open(log_file, 'w') as f:
    json.dump({
        'timestamp': timestamp,
        'dry_run': DRY_RUN,
        'verbose': VERBOSE,
        'batch_size': BATCH_SIZE,
        'skip_existing': SKIP_EXISTING,
        'update_existing': UPDATE_EXISTING,
        'total': len(monsters_to_sync),
        'created': results['created'],
        'updated': results['updated'],
        'skipped': results['skipped'],
        'errors': results['errors'],
        'details': results['details']
    }, f, indent=2)

print(f"\n✓ Log saved to: {log_file}")

# Print error summary if there were errors
if results['errors'] > 0 and not VERBOSE:
    print(f"\n⚠️  {results['errors']} errors occurred. Set VERBOSE = True for detailed output.")
    print(f"   Or check the log file for full details: {log_file}")



⚠️  LIVE MODE - Monsters WILL be created/updated!
Start time: 2025-12-25 13:50:21
Batch size: 1
Delay between requests: 1s
Verbose logging: ON
Skip existing: YES
Update existing: YES (will update when DDB ID exists)

Total monsters to process: 1

Checking spreadsheet for existing DDB IDs...
✓ Found 0 monsters with DDB IDs in spreadsheet

Fetching existing homebrew monsters from D&D Beyond...
✓ Found 5 existing homebrew monsters

[1/1] Processing: Pavani
  → Converting monster to D&D Beyond format...
    Type: 13, Size: 4
    CR: 7
  → Creating monster on D&D Beyond...
  ✗ Failed to create monster
    HTTP 200: OK

SYNC COMPLETE
✓ Created: 0
✓ Updated: 0
⚠️  Skipped (already exist): 0
✗ Errors: 1

ℹ️  Check the log file for detailed error information
End time: 2025-12-25 13:50:28

✓ Log saved to: out/monster_sync_log_20251225_135028.json


## 7. Batch Delete Monsters from D&D Beyond

**⚠️ DANGER ZONE - This will permanently delete monsters!**

Use this section to bulk delete monsters from D&D Beyond. This is useful for:
- Cleaning up duplicate monsters created by mistake
- Removing old versions after renaming monsters
- Starting fresh with a clean slate

**How it works:**
1. Fetches all your existing monsters from D&D Beyond
2. Allows you to filter monsters to delete (by name pattern, ID range, etc.)
3. Deletes the selected monsters one by one
4. Optionally clears DDB IDs from the spreadsheet

**Configuration:**
- `DRY_RUN`: Set to `False` to actually delete (always test with `True` first!)
- `DELETE_FILTER`: Function to filter which monsters to delete
- `CLEAR_SPREADSHEET_IDS`: Set to `True` to also remove DDB IDs from spreadsheet


In [8]:
# # ============================================
# # BATCH DELETE MONSTERS
# # ============================================
# 
# DRY_RUN = False  # ALWAYS test with True first!
# DELAY = 1  # Seconds between deletions
# VERBOSE = True  # Detailed logging
# CLEAR_SPREADSHEET_IDS = True  # Set to True to also clear DDB column in spreadsheet
# 
# # Define a filter function to select which monsters to delete
# # Examples:
# #   - Delete all monsters: lambda monster: True
# #   - Delete monsters with specific IDs: lambda monster: monster['id'] in ['6057264']
# #   - Delete monsters by name pattern: lambda monster: 'test' in monster['name'].lower()
# #   - Delete monsters with ID > X: lambda monster: int(monster['id']) > 6057000
# 
# DELETE_FILTER = lambda monster: True  # Default: delete nothing (safe)
# 
# # Example filters (uncomment one to use):
# # DELETE_FILTER = lambda monster: monster['id'] in ['6057264']  # Delete specific IDs
# # DELETE_FILTER = lambda monster: int(monster['id']) > 6057000  # Delete recent monsters
# # DELETE_FILTER = lambda monster: 'test' in monster['name'].lower()  # Delete by name
# 
# print('=' * 60)
# print('⚠️  DANGER: BATCH DELETE MONSTERS')
# print('=' * 60)
# if DRY_RUN:
#     print('DRY RUN - No monsters will be deleted')
# else:
#     print('⚠️  LIVE MODE - Monsters WILL BE PERMANENTLY DELETED!')
# print('=' * 60)
# print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
# print(f"Delay between deletions: {DELAY}s")
# print(f"Clear spreadsheet IDs: {'YES' if CLEAR_SPREADSHEET_IDS else 'NO'}")
# print()
# 
# # Fetch all existing monsters from D&D Beyond
# print('Fetching all homebrew monsters from D&D Beyond...')
# try:
#     all_monsters = ddb_api.monsters.list()
#     print(f"✓ Found {len(all_monsters)} existing homebrew monsters\n")
# except Exception as e:
#     print(f"✗ Error fetching monsters: {e}")
#     print('Cannot proceed with deletion')
#     import sys
#     sys.exit(1)
# 
# # Apply filter to select monsters to delete
# monsters_to_delete = [monster for monster in all_monsters if DELETE_FILTER(monster)]
# 
# print(f"Filter selected {len(monsters_to_delete)} monsters to delete")
# if monsters_to_delete:
#     print('Sample monsters to delete:')
#     for monster in monsters_to_delete[:5]:
#         print(f"  - {monster['name']} (ID: {monster['id']})")
#     if len(monsters_to_delete) > 5:
#         print(f"  ... and {len(monsters_to_delete) - 5} more")
# 
# if not monsters_to_delete:
#     print('No monsters selected for deletion. Exiting.')
# else:
#     results = {'deleted': 0, 'errors': 0}
# 
#     for i, monster in enumerate(monsters_to_delete, 1):
#         monster_id = monster['id']
#         monster_name = monster['name']
#         print(f"[{i}/{len(monsters_to_delete)}] Deleting: {monster_name} (ID: {monster_id})")
# 
#         if DRY_RUN:
#             print('  → DRY RUN: Would delete')
#             results['deleted'] += 1
#         else:
#             deleted = ddb_api.monsters.delete(monster_id)
#             if deleted:
#                 print('  ✓ Deleted successfully')
#                 results['deleted'] += 1
# 
#                 if CLEAR_SPREADSHEET_IDS:
#                     spreadsheet_updates = [{
#                         'match_value': monster_name,
#                         'update_column': 'DDB',
#                         'update_value': ''
#                     }]
#                     try:
#                         fantasy_sheets.batch_update_cells_by_row_match(
#                             MONSTERS_GID,
#                             MONSTER_NAME_COLUMN,
#                             spreadsheet_updates
#                         )
#                     except Exception as e:
#                         print(f"  ⚠️  Failed to clear spreadsheet ID: {e}")
#             else:
#                 print('  ✗ Failed to delete')
#                 if ddb_api.last_error:
#                     print(f"    Error: {ddb_api.last_error}")
#                 results['errors'] += 1
# 
#         if i < len(monsters_to_delete):
#             time.sleep(DELAY)
# 
#     print('\n' + '=' * 60)
#     print('DELETION COMPLETE')
#     print('=' * 60)
#     if DRY_RUN:
#         print(f"DRY RUN: Would delete {results['deleted']} monsters")
#     else:
#         print(f"✓ Deleted: {results['deleted']}")
#         print(f"✗ Errors: {results['errors']}")
#     print('=' * 60)
